### **PART 03. Modeling (PM2.5 Forecasting)**

This notebook builds **baselines + ML models** to predict `pm25`.

**Key constraints**
- **Time-series split**: Train = **2022–2025**, Test = **2026**
- **No shuffle**
- **Baselines are mandatory** to justify ML value


#### **0. Setup**

In [1]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

import xgboost as xgb

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42
TARGET = "pm25"

#### **A — Train/Test Split (time-series)**

- Train: 2022–2025
- Test: 2026
- No shuffle


In [2]:
# Robust project root: works whether you run from repo root or from notebooks/
_cwd = os.getcwd()
ROOT_DIR = os.path.dirname(_cwd) if os.path.basename(_cwd).lower() == "notebooks" else _cwd
DATA_PROCESSED = os.path.join(ROOT_DIR, "data", "processed")
DATA_FINAL = os.path.join(ROOT_DIR, "data", "final")

# Prefer imputed dataset; fallback to final; fallback to cleaned
candidates = [
    os.path.join(DATA_PROCESSED, "data_imputed.csv"),
    os.path.join(DATA_FINAL, "data_final.csv"),
    os.path.join(DATA_PROCESSED, "data_cleaned.csv"),
]
data_path = next((p for p in candidates if os.path.exists(p)), None)
if data_path is None:
    raise FileNotFoundError("No dataset found. Expected one of: " + ", ".join(candidates))

df = pd.read_csv(data_path)
if "timestamp_local" not in df.columns:
    raise ValueError("Expected a 'timestamp_local' column.")

df["timestamp_local"] = pd.to_datetime(df["timestamp_local"])
df = df.set_index("timestamp_local").sort_index()

display(df.head())
print("Loaded:", data_path)
print("Shape:", df.shape)
print("Date range:", df.index.min(), "->", df.index.max())
print("Columns:", list(df.columns))

,app_temp,azimuth,clouds,dewpt,dhi,dni,elev_angle,ghi,pod,precip,...,wind_dir,wind_gust_spd,wind_spd,aqi,co,no2,o3,pm10,pm25,so2
timestamp_local,,,,,,,,,,,,,,,,,,,,,
2022-01-13 00:00:00,17.0,239.9,100,15.9,0,0,-88.6,0,0,0.0,...,106,2.4,1.6,180.0,381.3,16.7,86.7,109.3,77.67,54.7
2022-01-13 01:00:00,16.8,95.5,100,16.3,0,0,-77.2,0,0,0.0,...,98,2.0,1.2,181.0,389.8,17.0,87.0,111.0,79.00,59.0
2022-01-13 02:00:00,16.4,96.7,91,16.0,0,0,-63.3,0,0,0.0,...,78,2.0,1.2,178.0,385.0,16.0,87.3,107.3,76.33,58.7
2022-01-13 03:00:00,16.1,99.2,83,15.6,0,0,-49.4,0,0,0.0,...,58,2.4,1.2,174.0,380.1,15.0,87.7,103.7,73.67,58.3
2022-01-13 04:00:00,15.7,102.2,75,15.2,0,0,-35.7,0,0,0.0,...,46,3.2,1.6,171.0,375.3,14.0,88.0,100.0,71.00,58.0


Loaded: c:\Users\admin\Downloads\Apps\ML air-quality-analysis-main\data\processed\data_imputed.csv
Shape: (36480, 27)
Date range: 2022-01-13 00:00:00 -> 2026-03-12 23:00:00
Columns: ['app_temp', 'azimuth', 'clouds', 'dewpt', 'dhi', 'dni', 'elev_angle', 'ghi', 'pod', 'precip', 'pres', 'rh', 'slp', 'solar_rad', 'temp', 'uv', 'vis', 'wind_dir', 'wind_gust_spd', 'wind_spd', 'aqi', 'co', 'no2', 'o3', 'pm10', 'pm25', 'so2']


In [3]:
# Basic cleaning: ensure numeric where expected; drop rows with missing target
df = df.copy()
df = df.dropna(subset=[TARGET])

# Feature set: use all numeric columns except the target
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
feature_cols = [c for c in numeric_cols if c != TARGET]

X = df[feature_cols]
y = df[TARGET]

# Time-based split
train_mask = (df.index.year >= 2022) & (df.index.year <= 2025)
test_mask = (df.index.year == 2026)

X_train, y_train = X.loc[train_mask], y.loc[train_mask]
X_test, y_test = X.loc[test_mask], y.loc[test_mask]

print("Train:", X_train.shape, y_train.shape, "years:", sorted(df.index[train_mask].year.unique().tolist()))
print("Test:", X_test.shape, y_test.shape, "years:", sorted(df.index[test_mask].year.unique().tolist()))

if len(X_test) == 0:
    raise ValueError("No rows found for test year 2026. Check your dataset date range.")

Train: (34776, 26) (34776,) years: [2022, 2023, 2024, 2025]
Test: (1704, 26) (1704,) years: [2026]


#### **B — Baseline Models**

Baselines used:
1. Mean predictor (predict constant mean of train)
2. Previous-hour predictor (\(\hat{y}_t = y_{t-1}\))


In [4]:
def mean_predictor(y_train: pd.Series, index: pd.DatetimeIndex) -> pd.Series:
    return pd.Series(y_train.mean(), index=index)

def prev_hour_predictor(y_all: pd.Series) -> pd.Series:
    # previous observed hour as naive forecast
    return y_all.shift(1)

yhat_mean_test = mean_predictor(y_train, y_test.index)

# Previous hour baseline needs full series to shift
yhat_prev_all = prev_hour_predictor(y)
yhat_prev_test = yhat_prev_all.loc[y_test.index]

print("Mean baseline sample:")
display(pd.DataFrame({"y": y_test.head(), "yhat_mean": yhat_mean_test.head(), "yhat_prev": yhat_prev_test.head()}))

Mean baseline sample:


,y,yhat_mean,yhat_prev
timestamp_local,,,
2026-01-01 00:00:00,49.0,50.081407,51.0
2026-01-01 01:00:00,69.5,50.081407,49.0
2026-01-01 02:00:00,80.8,50.081407,69.5
2026-01-01 03:00:00,81.2,50.081407,80.8
2026-01-01 04:00:00,82.0,50.081407,81.2


#### **E — Metrics**

Compute:
- RMSE
- MAE
- R²
- MAPE (optional)


In [5]:
def mape(y_true, y_pred, eps: float = 1e-8):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    denom = np.clip(np.abs(y_true), eps, None)
    return np.mean(np.abs((y_true - y_pred) / denom)) * 100

def compute_metrics(y_true: pd.Series, y_pred: pd.Series, include_mape: bool = True) -> dict:
    # align + drop NaNs
    dfm = pd.concat([y_true.rename("y"), y_pred.rename("yhat")], axis=1).dropna()
    y_t = dfm["y"].values
    y_p = dfm["yhat"].values
    out = {
        "RMSE": float(np.sqrt(mean_squared_error(y_t, y_p))),
        "MAE": float(mean_absolute_error(y_t, y_p)),
        "R2": float(r2_score(y_t, y_p)),
        "n": int(len(dfm)),
    }
    if include_mape:
        out["MAPE_%"] = float(mape(y_t, y_p))
    return out

baseline_results = {
    "MeanPredictor": compute_metrics(y_test, yhat_mean_test),
    "PrevHourPredictor": compute_metrics(y_test, yhat_prev_test),
}

pd.DataFrame(baseline_results).T

,RMSE,MAE,R2,n,MAPE_%
MeanPredictor,43.419219,29.952313,-0.150975,1704.0,55.121293
PrevHourPredictor,20.440929,11.227523,0.744904,1704.0,19.604982


#### **C — Train Models**

Models:
1. Linear Regression
2. Random Forest
3. XGBoost
4. (Optional) ARIMA (univariate)


In [6]:
def fit_predict_sklearn(model, X_train, y_train, X_test, index_test):
    model.fit(X_train, y_train)
    yhat = model.predict(X_test)
    return pd.Series(yhat, index=index_test)

# 1) Linear Regression
lin = LinearRegression(n_jobs=None)
yhat_lin = fit_predict_sklearn(lin, X_train, y_train, X_test, y_test.index)

# 2) Random Forest
rf = RandomForestRegressor(
    n_estimators=300,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
yhat_rf = fit_predict_sklearn(rf, X_train, y_train, X_test, y_test.index)

# 3) XGBoost
xgb_model = xgb.XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
yhat_xgb = fit_predict_sklearn(xgb_model, X_train, y_train, X_test, y_test.index)

model_results = {
    "LinearRegression": compute_metrics(y_test, yhat_lin),
    "RandomForest": compute_metrics(y_test, yhat_rf),
    "XGBoost": compute_metrics(y_test, yhat_xgb),
}

pd.DataFrame(model_results).T

,RMSE,MAE,R2,n,MAPE_%
LinearRegression,20.744513,9.621870,0.737271,1704.0,14.465261
RandomForest,6.626853,1.098158,0.973189,1704.0,1.182233
XGBoost,6.935933,2.204089,0.970629,1704.0,2.999806


#### **D — Hyperparameter Tuning (RF, XGBoost)**

Use `RandomizedSearchCV` (fast exploration) then optional `GridSearchCV` (fine search).

**Note:** This is time-series. For a stricter setup, replace CV with a time-series split (e.g., `TimeSeriesSplit`).


In [ ]:
from sklearn.model_selection import TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=5)

# --- RF RandomizedSearch ---
rf_base = RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1)
rf_param_dist = {
    "n_estimators": [200, 400, 600],
    "max_depth": [None, 10, 20, 30],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", 0.5, 0.8],
}

rf_rs = RandomizedSearchCV(
    rf_base,
    rf_param_dist,
    n_iter=20,
    scoring="neg_root_mean_squared_error",
    cv=tscv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
)

rf_rs.fit(X_train, y_train)
print("RF best params:", rf_rs.best_params_)
print("RF best CV RMSE:", -rf_rs.best_score_)

yhat_rf_tuned = pd.Series(rf_rs.best_estimator_.predict(X_test), index=y_test.index)
tuned_results = {
    "RandomForest_tuned": compute_metrics(y_test, yhat_rf_tuned)
}
pd.DataFrame(tuned_results).T

In [ ]:
# --- XGBoost RandomizedSearch ---
xgb_base = xgb.XGBRegressor(
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

xgb_param_dist = {
    "n_estimators": [300, 600, 1000],
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth": [3, 5, 7, 9],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "reg_lambda": [0.5, 1.0, 2.0],
    "min_child_weight": [1, 5, 10],
}

xgb_rs = RandomizedSearchCV(
    xgb_base,
    xgb_param_dist,
    n_iter=25,
    scoring="neg_root_mean_squared_error",
    cv=tscv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
)

xgb_rs.fit(X_train, y_train)
print("XGB best params:", xgb_rs.best_params_)
print("XGB best CV RMSE:", -xgb_rs.best_score_)

yhat_xgb_tuned = pd.Series(xgb_rs.best_estimator_.predict(X_test), index=y_test.index)
pd.DataFrame({"XGBoost_tuned": compute_metrics(y_test, yhat_xgb_tuned)}).T

#### **F — Compare Models**

Table: Model / RMSE / MAE / R²


In [ ]:
all_results = {}
all_results.update(baseline_results)
all_results.update(model_results)
all_results["RandomForest_tuned"] = compute_metrics(y_test, yhat_rf_tuned)
all_results["XGBoost_tuned"] = compute_metrics(y_test, yhat_xgb_tuned)

compare = pd.DataFrame(all_results).T
compare = compare[["RMSE", "MAE", "R2", "MAPE_%", "n"]]
compare.sort_values(by="RMSE")

#### **G — Feature Importance / SHAP**

Use SHAP summary plot on the best-performing tree model (typically tuned XGBoost).


In [ ]:
import shap

# pick best estimator (you can change this if RF wins)
best_tree = xgb_rs.best_estimator_

# SHAP can be expensive; sample the test set
SAMPLE_N = min(2000, len(X_test))
X_sample = X_test.sample(SAMPLE_N, random_state=RANDOM_STATE) if len(X_test) > SAMPLE_N else X_test

explainer = shap.TreeExplainer(best_tree)
shap_values = explainer.shap_values(X_sample)

shap.summary_plot(shap_values, X_sample, show=False)
plt.tight_layout()
plt.show()

#### **H — Error Analysis**

Examples to investigate:
1. Larger errors during heavy rain
2. Larger errors during sudden PM2.5 spikes
3. Larger errors at night / rush hours


In [ ]:
# Choose which model predictions to analyze
yhat_best = yhat_xgb_tuned

err_df = pd.DataFrame({
    "y": y_test,
    "yhat": yhat_best,
})
err_df["abs_error"] = (err_df["y"] - err_df["yhat"]).abs()
err_df["error"] = err_df["y"] - err_df["yhat"]
err_df["hour"] = err_df.index.hour
err_df["month"] = err_df.index.month

# 1) Heavy rain (if precip exists)
if "precip" in df.columns:
    err_df["precip"] = df.loc[err_df.index, "precip"]
    heavy = err_df[err_df["precip"] >= err_df["precip"].quantile(0.95)]
    print("Heavy rain rows (top 5% precip):", len(heavy))
    display(heavy.sort_values("abs_error", ascending=False).head(10))

# 2) PM2.5 spikes (top 1% of true values)
spike = err_df[err_df["y"] >= err_df["y"].quantile(0.99)]
print("Spike rows (top 1% PM2.5):", len(spike))
display(spike.sort_values("abs_error", ascending=False).head(10))

# 3) Night vs day error profile
err_by_hour = err_df.groupby("hour")["abs_error"].mean()
plt.figure(figsize=(10, 3))
err_by_hour.plot(kind="bar")
plt.title("Mean absolute error by hour of day (test year)")
plt.xlabel("Hour")
plt.ylabel("Mean |error|")
plt.tight_layout()
plt.show()

# Visualize actual vs predicted for a sample window
window = err_df.sort_index().iloc[:24*14]  # first 2 weeks of test
plt.figure(figsize=(12, 4))
plt.plot(window.index, window["y"], label="Actual", linewidth=2)
plt.plot(window.index, window["yhat"], label="Predicted", alpha=0.8)
plt.title("Actual vs Predicted PM2.5 (sample window)")
plt.legend()
plt.tight_layout()
plt.show()

#### **(Optional) ARIMA (Univariate baseline/model)**

ARIMA uses only the target series. This is optional and can be slow; keep it as a separate experiment.


In [ ]:
from statsmodels.tsa.arima.model import ARIMA

# Fit on train only
y_train_series = y_train.asfreq("H")  # ensure hourly freq (may introduce NaNs)
y_train_series = y_train_series.interpolate(limit_direction="both")

# Simple ARIMA order (p,d,q). You can tune this later.
arima_order = (2, 0, 2)
arima = ARIMA(y_train_series, order=arima_order)
arima_fit = arima.fit()

# Forecast length = test set length
steps = len(y_test)
arima_forecast = arima_fit.forecast(steps=steps)
arima_forecast.index = y_test.index

print("ARIMA metrics:")
display(pd.DataFrame({"ARIMA": compute_metrics(y_test, arima_forecast)}).T)